# Module 9: RAG & LLM Applications
## 03: Production API Integration, Retries & Resilience

### Theoretical Overview
Production LLM calls must be fault-tolerant against transient network partitions, HTTP 429 rate limits, and server-side timeouts. Key best practices include:
1. **Exponential Backoff with Jitter**: Avoids thundering herd problems when retrying failed requests.
2. **Circuit Breakers**: Gracefully stops invoking degraded upstream endpoints.
3. **Async / Concurrent Fan-Out**: Processing batches of document chunks asynchronously.

In [ ]:
import time
import random
from typing import Callable, Any

def retry_with_exponential_backoff(
    func: Callable[[], Any],
    max_retries: int = 4,
    base_delay: float = 0.5,
    max_delay: float = 10.0
) -> Any:
    """Executes a callable with exponential backoff and randomized jitter."""
    retries = 0
    while True:
        try:
            return func()
        except Exception as e:
            retries += 1
            if retries > max_retries:
                raise RuntimeError(f"Max retries ({max_retries}) exceeded: {e}")
            jitter = random.uniform(0.1, 0.5)
            delay = min(max_delay, (base_delay * (2 ** (retries - 1)))) + jitter
            print(f"[Attempt {retries}] Caught: '{e}'. Retrying in {delay:.2f}s...")
            time.sleep(delay)

# Demonstration with simulated rate limit failure
attempt_counter = 0
def mock_flaky_api_call():
    global attempt_counter
    attempt_counter += 1
    if attempt_counter < 3:
        raise ConnectionResetError("429 Too Many Requests: Rate limit exceeded")
    return {"status": "success", "response": "API request completed successfully!"}

result = retry_with_exponential_backoff(mock_flaky_api_call)
print("Final result:", result)

### Async Multi-Prompt Dispatching Pattern

```python
import asyncio
from typing import List

async def process_batch(prompts: List[str]):
    tasks = [async_call_llm(p) for p in prompts]
    return await asyncio.gather(*tasks)
```

In [ ]:
import asyncio

async def mock_async_llm_worker(prompt_id: int, prompt_text: str):
    await asyncio.sleep(0.05)
    return {"prompt_id": prompt_id, "tokens": len(prompt_text.split()), "status": "processed"}

async def run_batch():
    prompts = ["Analyze Q3 sales", "Summarize contract 402", "Extract invoice items"]
    results = await asyncio.gather(*(mock_async_llm_worker(i, p) for i, p in enumerate(prompts)))
    return results

# Run demo in current event loop
import nest_asyncio
nest_asyncio.apply()
batch_out = asyncio.run(run_batch())
print("Processed batch output:")
print(json.dumps(batch_out, indent=2))